# Julia AOT plugins

Julia is normally JIT-compiled: a fresh `julia` process compiles code the
first time it runs, which is fine for interactive work but harmful for a
propagator's hot loop. propaq needs a plain C-ABI shared library it can
`dlopen` and call into with zero Julia runtime startup cost. `PackageCompiler.jl`
bridges that gap: it ahead-of-time compiles Julia source (plus a minimal
Julia runtime) into a standalone `.so` that behaves like any other C
library from the caller's side. This allows propaq to call into Julia code with no JIT overhead.


## The plugin files are bare scripts; `create_library` needs a package

Every `.jl` file under `examples/plugins/julia/` is a standalone script, matching the single-file
convention the C and Rust examples use. `PackageCompiler.create_library`
does not accept that directly, rather it expects a directory containing a Julia
*package* (a `Project.toml` plus a `src/<Name>.jl` entry point).

In [ ]:
import subprocess

result = subprocess.run(
    [
        "julia", "-e",
        '''
        using PackageCompiler
        create_library("../julia/noise", "/tmp/_propaq_aot_probe")
        ''',
    ],
    capture_output=True, text=True, timeout=30,
)
print("exit code:", result.returncode)
print(result.stderr.strip().splitlines()[0])

exit code: 1
ERROR: could not find project at "../julia/noise"


## Wrapping a plugin in a minimal package

No changes to the canonical plugin file are needed, the wrapper package's
entry point just `include()`s it unmodified inside a `module` so that it's recognized
by the Julia compiler. This keeps
the single-file plugin as the thing you read and edit, and treats the
package wrapper as a build-time-only artifact.

In [ ]:
from pathlib import Path

PLUGIN_SRC = Path("../julia/noise/thermal_decay_noise.jl").resolve()

PKG_DIR = Path("_build/ThermalDecayNoisePlugin").resolve()
(PKG_DIR / "src").mkdir(parents=True, exist_ok=True)

(PKG_DIR / "Project.toml").write_text('''\
name = "ThermalDecayNoisePlugin"
uuid = "8f5b3b1e-1234-4a2b-9c3d-abc123456789"
version = "0.1.0"
''')

(PKG_DIR / "src" / "ThermalDecayNoisePlugin.jl").write_text(f'''\
module ThermalDecayNoisePlugin

include("{PLUGIN_SRC}")

end # module
''')

print("Wrapper package written to", PKG_DIR)
for p in sorted(PKG_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(PKG_DIR))

## Building the library

```julia
using PackageCompiler
create_library(
    "_build/ThermalDecayNoisePlugin",   # the wrapper package from above
    "_build/thermal_decay_noise_lib";   # output directory
    lib_name="thermal_decay_noise",
    force=true,
    filter_stdlibs=true,                # smaller sysimage: skip unused stdlibs
)
```

The result is `_build/thermal_decay_noise_lib/lib/libthermal_decay_noise.so`
(`.dylib` on macOS), loadable exactly like the C and Rust `.so` files in the
other two notebooks:

```python
from propaq.noise import NativeNoiseModel
model = NativeNoiseModel(
    "_build/thermal_decay_noise_lib/lib/libthermal_decay_noise.so",
    config='{"gamma": 0.02, "beta": 1.5}',
)
```

## Verifying the Julia logic without paying for a full AOT build

The `@ccallable` wrappers exist for the C ABI; the actual math is plain,
ordinary Julia functions underneath (`damping_factor`, `keep`, ...), so they
can be called directly.

In [3]:
import subprocess

result = subprocess.run(
    [
        "julia", "-e",
        '''
        include("../julia/noise/thermal_decay_noise.jl")
        for w in UInt32[0, 1, 3, 7, 12]
            println(damping_factor(0.02, 1.7, w))
        end
        ''',
    ],
    capture_output=True, text=True, timeout=30, cwd=".",
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

1.0
0.9987073821379409
0.9916624141143802
0.9652652944341173
0.9154120010726767



matching the values from the first two notebooks.